# Lid-driven cavity (Navier–Stokes)

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/lid_driven_cavity)

The classic benchmark of computational fluid dynamics: a square box of fluid whose lid slides to the right. The 2D Navier–Stokes equations in streamfunction–vorticity form are marched in time, with the Poisson equation for $\psi$ solved by diagonalising the Laplacian with eigh. A primary vortex spins up and its centreline velocity is compared with Ghia et al. (1982).

**Vector form**

$$\partial_t\omega + (\mathbf u\cdot\nabla)\,\omega = \nu\,\nabla^2\omega,\qquad \nabla^2\psi = -\omega,\qquad \mathbf u = (\partial_y\psi,\,-\partial_x\psi)$$

**Index (tensor) form**

$$\partial_t\omega + u_j\,\partial_j\omega = \nu\,\partial_j\partial_j\omega,\qquad \partial_j\partial_j\psi = -\omega,\qquad u_i = \epsilon_{ij}\,\partial_j\psi \;\Rightarrow\; \partial_i u_i = 0$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\omega_{xy},\ \psi_{xy}$ | `omega, psi` | (N, N) | vorticity and streamfunction on the grid, indexed [x, y], walls included |
| $u_j$ | `velocity(psi, h)` | (2, N-2, N-2) | $(\partial_y\psi, -\partial_x\psi)$ by central differences |
| $u_j\,\partial_j\omega$ | `einsum('jxy,jxy->xy', u, grad_omega)` | (N-2, N-2) | repeated $j$ ⇒ summed: advection of vorticity |
| $\partial_j\partial_j\omega$ | `laplacian(omega, h)` | (N-2, N-2) | five-point stencil |
| $T = V\Lambda V^\mathsf T$ | `laplacian_eigen(n, h, dev)` | (n, n), (n,) | 1D Laplacian with $\psi = 0$ walls, diagonalised once |
| $\hat f_{ab} = V_{ia}f_{ij}V_{jb}$ | `einsum('ia,ij,jb->ab', V, f, V)` | (n, n) | transform; then $\hat\psi_{ab} = \hat f_{ab}/(\Lambda_a + \Lambda_b)$ and back |
| $\omega_\text{wall}$ | `wall_vorticity(omega, psi, h, lid)` | (N, N) | Thom: $-2\psi_\text{next}/h^2$, minus $2U/h$ under the lid |

Units: cavity size 1 and lid speed 1, so $\mathrm{Re} = 1/\nu$. The streamfunction makes the flow divergence-free by construction, and the pressure disappears by taking the curl of Navier–Stokes. Each step: vorticity on the walls from Thom's formula (no slip), a forward Euler step for $\omega$, then the Poisson solve. $\nabla^2\psi = T\psi + \psi T^\mathsf T$ becomes diagonal in the eigenbasis of $T$ — the same idea as a fast sine transform. At Re = 100 the vortex centre sits near (0.62, 0.74), above and right of the centre; at lower Re it moves to (0.5, 0.76), at higher Re towards the middle, and small counter-rotating eddies appear in the bottom corners. Higher Re needs a finer grid.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/lid_driven_cavity/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import divergence, laplacian, velocity

LID = 1.0  # lid speed; with cavity size 1 the Reynolds number is Re = U L / ν = 1/ν


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, re = params["n"], params["re"]
    nu = LID / re
    y, h = tensors.grid(n, dev)  # (N,)
    V, lam = tensors.laplacian_eigen(n - 2, h, dev)  # (n, n), (n,)
    omega, _ = tensors.at_rest(n, dev)  # (N, N)

    times, omegas, psis = solve.simulate(omega, V, lam, h, nu, LID, params["t_max"], save_every=10)
    u_centre = torch.stack([solve.centreline_u(p, h) for p in psis])  # (F, N)

    psi = psis[-1]
    u = velocity(psi, h)  # (2, N-2, N-2)
    residual = laplacian(psi, h) + omegas[-1][1:-1, 1:-1]
    i, j = divmod(int(psi.argmin()), n)
    print(f"{n}x{n} grid, Re = {re:.0f}, dt = {solve.time_step(h, nu, LID):.4f}, t = {times[-1]:.1f}")
    print(
        f"Poisson residual max |∇²ψ + ω| = {residual.abs().max():.1e};  "
        f"max |∂_i u_i| = {divergence(u, h).abs().max():.1e}"
    )
    print(
        f"primary vortex centre ({y[i]:.3f}, {y[j]:.3f}), ψ_min = {psi.min():.4f}  "
        "(Ghia Re=100: (0.617, 0.734), -0.1034)"
    )
    ghia = None
    if abs(re - 100) < 1e-9:
        ghia = solve.GHIA_RE100
        print(f"centreline u vs Ghia et al.: max deviation {solve.ghia_deviation(y, u_centre[-1]):.3f}")
    return plot.animate(y, times, omegas, psis, u_centre, ghia, re, out_dir / "lid_driven_cavity.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/lid_driven_cavity/equation.py
r"""Physics → tensor translation.

2D incompressible Navier–Stokes in streamfunction–vorticity form. With a streamfunction ψ,

    u = ∂_y ψ,   v = -∂_x ψ     ⇒   ∂_i u_i = 0 automatically,

and the vorticity ω = ∂_x v - ∂_y u obeys a transport equation plus a Poisson equation for ψ:

    ∂_t ω + u_j ∂_j ω = ν ∂_j ∂_j ω          (j summed: advection and diffusion)
    ∂_j ∂_j ψ = -ω

On the grid f[i, j] = f(x_i, y_j) the derivatives are central differences with spacing h.
The Poisson equation is solved with the eigenvectors of the 1D Laplacian matrix T
(T = V Λ V^T, see solve.py): ∇²ψ = T ψ + ψ T^T becomes diagonal, (Λ_a + Λ_b) \hat ψ_{ab} = \hat f_{ab}.
"""

import torch


def ddx(f: torch.Tensor, h: float) -> torch.Tensor:
    """Central ∂_x on interior points: (N, N) → (N-2, N-2)."""
    return (f[2:, 1:-1] - f[:-2, 1:-1]) / (2 * h)


def ddy(f: torch.Tensor, h: float) -> torch.Tensor:
    """Central ∂_y on interior points: (N, N) → (N-2, N-2)."""
    return (f[1:-1, 2:] - f[1:-1, :-2]) / (2 * h)


def laplacian(f: torch.Tensor, h: float) -> torch.Tensor:
    r"""∂_j ∂_j f, five-point stencil on interior points: (N, N) → (N-2, N-2)."""
    return (f[2:, 1:-1] + f[:-2, 1:-1] + f[1:-1, 2:] + f[1:-1, :-2] - 4 * f[1:-1, 1:-1]) / h**2


def velocity(psi: torch.Tensor, h: float) -> torch.Tensor:
    """u_i = (∂_y ψ, -∂_x ψ) on interior points: (N, N) → (2, N-2, N-2)."""
    return torch.stack([ddy(psi, h), -ddx(psi, h)])


def divergence(u: torch.Tensor, h: float) -> torch.Tensor:
    """∂_i u_i by central differences: (2, M, M) → (M-2, M-2). Zero to round-off for u from a streamfunction."""
    return (u[0, 2:, 1:-1] - u[0, :-2, 1:-1] + u[1, 1:-1, 2:] - u[1, 1:-1, :-2]) / (2 * h)


def vorticity_rhs(omega: torch.Tensor, psi: torch.Tensor, h: float, nu: float) -> torch.Tensor:
    r"""∂_t ω = -u_j ∂_j ω + ν ∂_j ∂_j ω  (j summed by einsum).   (N, N) fields → (N-2, N-2)."""
    u = velocity(psi, h)  # (2, N-2, N-2)
    grad_omega = torch.stack([ddx(omega, h), ddy(omega, h)])  # (2, N-2, N-2)
    advection = torch.einsum("jxy,jxy->xy", u, grad_omega)  # u_j ∂_j ω
    return -advection + nu * laplacian(omega, h)


def wall_vorticity(omega: torch.Tensor, psi: torch.Tensor, h: float, lid: float) -> torch.Tensor:
    """Thom's formula: vorticity on the walls from ψ next to them (ψ = 0 on the walls). Returns a new ω.

    Bottom, left, right walls are at rest: ω_wall = -2 ψ_next / h².  The top lid moves at speed U:
    ω_top = -2 ψ_next / h² - 2 U / h.
    """
    omega = omega.clone()
    omega[:, 0] = -2 * psi[:, 1] / h**2  # bottom
    omega[0, :] = -2 * psi[1, :] / h**2  # left
    omega[-1, :] = -2 * psi[-2, :] / h**2  # right
    omega[:, -1] = -2 * psi[:, -2] / h**2 - 2 * lid / h  # top (moving lid)
    return omega


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/lid_driven_cavity/tensors.py
"""Build the concrete tensors: the square grid, the eigen-decomposed 1D Laplacian and the initial fields."""

import torch


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """N points per side on [0, 1] (walls included) and the spacing h = 1/(N-1)."""
    return torch.linspace(0.0, 1.0, n, device=device, dtype=torch.float64), 1.0 / (n - 1)


def laplacian_eigen(n_interior: int, h: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """1D Dirichlet Laplacian T = tridiag(1, -2, 1)/h² on the interior points, decomposed as T = V diag(Λ) V^T.

    Returns V (n, n) and Λ (n,). Done once; every Poisson solve then costs four small matrix products.
    """
    T = (
        torch.diag(torch.full((n_interior - 1,), 1.0), 1)
        + torch.diag(torch.full((n_interior,), -2.0))
        + torch.diag(torch.full((n_interior - 1,), 1.0), -1)
    ).to(device=device, dtype=torch.float64) / h**2
    lam, V = torch.linalg.eigh(T)
    return V, lam


def at_rest(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """ω = ψ = 0 everywhere: two (N, N) tensors indexed [i (x), j (y)]."""
    zeros = torch.zeros(n, n, device=device, dtype=torch.float64)
    return zeros.clone(), zeros.clone()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/lid_driven_cavity/solve.py
"""Time-march vorticity, solving the Poisson equation for ψ at every step."""

import numpy as np
import torch

from problems._common.progress import steps

from .equation import vorticity_rhs, wall_vorticity

# Ghia, Ghia & Shin (1982), Re = 100: u along the vertical centreline x = 0.5 (y, u)
GHIA_RE100 = torch.tensor(
    [[1.0000, 1.00000], [0.9766, 0.84123], [0.9688, 0.78871], [0.9609, 0.73722], [0.9531, 0.68717],
     [0.8516, 0.23151], [0.7344, 0.00332], [0.6172, -0.13641], [0.5000, -0.20581], [0.4531, -0.21090],
     [0.2813, -0.15662], [0.1719, -0.10150], [0.1016, -0.06434], [0.0703, -0.04775], [0.0625, -0.04192],
     [0.0547, -0.03717], [0.0000, 0.00000]], dtype=torch.float64,
)  # fmt: skip


def poisson_solve(f: torch.Tensor, V: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    r"""Solve ∂_j∂_j ψ = f with ψ = 0 on the walls.   f: interior (n, n) → full ψ (n+2, n+2).

    \hat f_{ab} = V_{ia} f_{ij} V_{jb},   \hat ψ_{ab} = \hat f_{ab} / (Λ_a + Λ_b),   ψ_{ij} = V_{ia} \hat ψ_{ab} V_{jb}
    """
    f_hat = torch.einsum("ia,ij,jb->ab", V, f, V)
    psi_hat = f_hat / (lam[:, None] + lam[None, :])
    psi = torch.einsum("ia,ab,jb->ij", V, psi_hat, V)
    return torch.nn.functional.pad(psi, (1, 1, 1, 1))  # add the walls, where ψ = 0


def time_step(h: float, nu: float, lid: float) -> float:
    """Forward Euler with central differences: stable below h²/(4ν) (diffusion) and h/U (advection)."""
    return 0.5 * min(h**2 / (4 * nu), h / lid)


def simulate(
    omega: torch.Tensor, V: torch.Tensor, lam: torch.Tensor, h: float, nu: float, lid: float, t_max: float,
    save_every: int = 20,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:  # fmt: skip
    """Return (times (F,), ω snapshots (F, N, N), ψ snapshots (F, N, N))."""
    dt = time_step(h, nu, lid)
    n_steps = int(t_max / dt)
    psi = poisson_solve(-omega[1:-1, 1:-1], V, lam)
    times, omegas, psis = [0.0], [omega], [psi]
    for n in steps(n_steps, "time steps"):
        omega = wall_vorticity(omega, psi, h, lid)
        omega[1:-1, 1:-1] = omega[1:-1, 1:-1] + dt * vorticity_rhs(omega, psi, h, nu)  # forward Euler
        psi = poisson_solve(-omega[1:-1, 1:-1], V, lam)  # ∇²ψ = -ω
        if (n + 1) % save_every == 0:
            times.append((n + 1) * dt)
            omegas.append(omega.clone())
            psis.append(psi)
    return torch.tensor(times), torch.stack(omegas), torch.stack(psis)


def centreline_u(psi: torch.Tensor, h: float) -> torch.Tensor:
    """u = ∂_y ψ along x = 0.5 (N odd, so the centre is a grid line); includes the walls: (N,)."""
    mid = psi.shape[0] // 2
    u = torch.zeros(psi.shape[1], dtype=psi.dtype, device=psi.device)
    u[1:-1] = (psi[mid, 2:] - psi[mid, :-2]) / (2 * h)
    u[-1] = 1.0  # the lid
    return u


def ghia_deviation(y: torch.Tensor, u: torch.Tensor) -> float:
    """Largest |u - u_Ghia| at the Ghia et al. points (linear interpolation of our centreline u)."""
    u_here = np.interp(GHIA_RE100[:, 0].numpy(), y.cpu().numpy(), u.cpu().numpy())
    return float(np.abs(u_here - GHIA_RE100[:, 1].numpy()).max())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/lid_driven_cavity/plot.py
"""Left: vorticity with streamlines (contours of ψ) and the vortex centre. Right: u on the vertical centreline."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    y: torch.Tensor, times: torch.Tensor, omega: torch.Tensor, psi: torch.Tensor, u_centre: torch.Tensor,
    ghia: torch.Tensor | None, re: float, path: Path,
) -> Path:  # fmt: skip
    ys, t = y.cpu().numpy(), times.cpu().numpy()
    w, p, uc = omega.cpu().numpy(), psi.cpu().numpy(), u_centre.cpu().numpy()  # (F, N, N), (F, N, N), (F, N)
    levels = np.concatenate([-np.geomspace(1e-4, 0.1, 9)[::-1], np.geomspace(1e-6, 1e-4, 3)])  # ψ < 0: main vortex

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.9), gridspec_kw={"width_ratios": [1.25, 1]})
    image = ax.imshow(
        w[0].T, origin="lower", extent=(0, 1, 0, 1), cmap=plt.get_cmap("RdBu_r", 15), vmin=-6, vmax=6,
        interpolation="nearest",
    )  # fmt: skip
    fig.colorbar(image, ax=ax, label="vorticity ω", shrink=0.8)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(False)
    centre = ax.plot([], [], "x", color="#f59e0b", ms=9, mew=2)[0]
    title = ax.set_title(" ")
    contours = []

    ax2.plot([0, 0], [0, 1], color="#9ca3af", lw=0.8)
    if ghia is not None:
        g = ghia.cpu().numpy()
        ax2.plot(g[:, 1], g[:, 0], "o", mfc="none", color="#111827", ms=5, label="Ghia et al. 1982")
    line = ax2.plot(uc[0], ys, color="#2563eb", lw=2, label="this solver")[0]
    ax2.set_xlim(-0.5, 1.05)
    ax2.set_ylim(0, 1)
    ax2.set_xlabel("u(x = 0.5, y)")
    ax2.set_ylabel("y")
    ax2.legend(loc="lower right", fontsize=8)

    def update(step: int):
        image.set_data(w[step].T)
        for c in contours:
            c.remove()
        contours.clear()
        contours.append(ax.contour(ys, ys, p[step].T, levels=levels, colors="k", linewidths=0.6))
        i, j = np.unravel_index(np.argmin(p[step]), p[step].shape)
        centre.set_data([ys[i]], [ys[j]])
        line.set_xdata(uc[step])
        title.set_text(f"lid → U = 1    Re = {re:.0f}    t = {t[step]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    're': 100.0,
    'n': 65,
    't_max': 20.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.lid_driven_cavity")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.lid_driven_cavity.main")
problem_dir = Path("problems/fluids/lid_driven_cavity")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
